In [1]:
import pandas as pd

llamamos a la url del primer archivo: asociasion

In [2]:
url="https://raw.githubusercontent.com/2516602022/Parcial_4-Elmer_Menendez-2516602022/refs/heads/main/dataset/clave_G_asociacion.csv"

Cargamos el archivo

In [3]:
df = pd.read_csv(url)

vemos las primeras filas del archivo

In [4]:
display(df.head())

,transaccion_id,cliente_id,fecha,categoria,item,cantidad,canal
0,G-T0001,G-C0072,2026-02-20,Construccion,Arena,1,Tienda
1,G-T0001,G-C0072,2026-02-20,Herramientas,Destornillador,1,Tienda
2,G-T0001,G-C0072,2026-02-20,Pintura,Lija,3,Tienda
3,G-T0002,G-C0057,2026-01-26,Electricidad,Cable,2,App
4,G-T0002,G-C0057,2026-01-26,Electricidad,Interruptor,1,App


como se puede ver la estructura del archivo es algo comun, se miran las transacciones hechas en la tienda, y sus datos fundamentales como el cliente identificado con su id, la fecha de la transaccion, las categorias de los productos y el producto en si como el item, tambien la cantidad de cada producto que se vendio y finalmente si la venta se hizo fisicamente en la tienda o en la app de la tienda

In [7]:
print(df.isnull().sum())

transaccion_id    0
cliente_id        0
fecha             0
categoria         0
item              0
cantidad          0
canal             1
dtype: int64


como podemos verificar hay un unico valor nulo en la columna canal, lo que quiere decir que hay una transaccion que no se sabe si se hizo en la tienda o desde la app

In [8]:
print(df.dtypes)

transaccion_id    object
cliente_id        object
fecha             object
categoria         object
item              object
cantidad           int64
canal             object
dtype: object


como podemos observar todas las columnas son de tipo object, menos una que es la cantidad la cual es int64

In [9]:
print(df.duplicated().sum())

1


solo hay un elemento duplicado

In [13]:
df = df.dropna()
df = df.drop_duplicates()

hacemos la respectiva limpieza de datos para aplicar las reglas asociacion

In [14]:
cesta = (df.groupby(['transaccion_id', 'item'])['cantidad']
         .sum().unstack().reset_index().fillna(0)
         .set_index('transaccion_id'))

aqui agrupamos por transaccion y por el producto, esto sumando cantidades, unstac convierte los productos en columnas

In [16]:
def codificar_unidades(x):
    if x <= 0:
        return False
    if x >= 1:
        return True

arriba entonces creamos una funcion oara que los numeros convertirlos a V o F dandoles un formato booleano

In [17]:
cesta_codificada = cesta.map(codificar_unidades)

print("\n--- MATRIZ TRANSACCIONAL LISTA PARA APRIORI ---")
print(cesta_codificada.head())


--- MATRIZ TRANSACCIONAL LISTA PARA APRIORI ---
item            Alicate  Arena  Brocha  Cable  Cemento  Clavos   Codo  \
transaccion_id                                                          
G-T0001           False   True   False  False    False   False  False   
G-T0002           False  False   False   True    False   False  False   
G-T0003           False   True    True  False    False   False  False   
G-T0004           False  False   False  False    False   False  False   
G-T0005           False   True   False   True     True   False  False   

item            Destornillador   Foco  Interruptor   Lija  Llave_paso  \
transaccion_id                                                          
G-T0001                   True  False        False   True       False   
G-T0002                  False  False         True  False        True   
G-T0003                  False  False        False  False       False   
G-T0004                  False  False        False  False        True   
G

finalmente  aplicamos la funcion a la matriz y imprimimos para observar los datos, de esta forma podemos ver que productos si se compraron en determinadas transacciones

In [18]:
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


ahora buscamos aplicar apriori, primero importamos las herramientas de la libreria mixtend

In [20]:
itemsets_frecuentes = apriori(cesta_codificada, min_support=0.03, use_colnames=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

buscamos combinaciones de productos que aparezcan juntos, almenos en el 3 por ciento de las transacciones

In [22]:
reglas = association_rules(itemsets_frecuentes, metric="lift", min_threshold=1.0)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

calculamos confianza y el lifd de esas combinaciones, nos quedamos solo con los que sean mayor a 1

In [25]:
top_10_reglas = reglas.sort_values(by=['lift', 'confidence'], ascending=[False, False]).head(10)

print("--- LAS 10 REGLAS DE ASOCIACIÓN MÁS RELEVANTES ---")
print(top_10_reglas[['antecedents', 'consequents', 'support', 'confidence', 'lift']].round(3))

--- LAS 10 REGLAS DE ASOCIACIÓN MÁS RELEVANTES ---
                  antecedents                consequents  support  confidence  \
80          (Rodillo, Brocha)           (Pintura_blanca)    0.092       0.900   
81           (Pintura_blanca)          (Rodillo, Brocha)    0.092       0.383   
79   (Pintura_blanca, Brocha)                  (Rodillo)    0.092       0.818   
82                  (Rodillo)   (Pintura_blanca, Brocha)    0.092       0.391   
39              (Interruptor)           (Destornillador)    0.036       0.333   
38           (Destornillador)              (Interruptor)    0.036       0.350   
78  (Pintura_blanca, Rodillo)                   (Brocha)    0.092       0.750   
83                   (Brocha)  (Pintura_blanca, Rodillo)    0.092       0.375   
64                 (Tubo_PVC)            (Pegamento_PVC)    0.128       0.625   
65            (Pegamento_PVC)                 (Tubo_PVC)    0.128       0.568   

     lift  
80  3.734  
81  3.734  
79  3.468  
82  3.468

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

ahiora mostramos las 10 mejores, ordenadas de mayor a menor importancia, usando primero lift y despues la confianza